# Prep

In [ ]:
import mne #eeg analysis
import numpy as np # dataframes and staff
import matplotlib.pyplot as plt # plots
import matplotlib # plots
import PyQt5 # how plots open in you PC
from pathlib import Path # path to files
from mne.channels import find_ch_adjacency # connection btw electrodes
from functools import partial # statistics
import scipy.stats #statistics
import pickle # way to save data
import seaborn as sns # data and stats
from scipy import stats # stats
import statistics # stats
import mne_qt_browser
from mne.datasets import somato
import pathlib
import os
from scipy.stats import wilcoxon
from scipy.stats import ttest_rel
from mne_connectivity import spectral_connectivity_time

#the following magic line will make all of your figures to be of the chosen size (in inches)
%matplotlib qt5

mne.viz.set_browser_backend('qt')

import pickle

Set parameters

In [ ]:
path_to_saves = Path("/Users/alexmedvedev/Desktop/Vishka/saves_useless")
path_to_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/epoches_useless")
path_to_exp_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/exp_epochs")
path_to_contr_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/contr_epochs")

In [ ]:
path_to_80_contr = Path('/Users/alexmedvedev/Desktop/Vishka/80_contr')
path_to_80_exp = Path('/Users/alexmedvedev/Desktop/Vishka/80_exp')
path_to_80_saves = Path('/Users/alexmedvedev/Desktop/Vishka/80_saves')
path_to_80_all = Path('/Users/alexmedvedev/Desktop/Vishka/80_all_epoches')

In [ ]:
path_to_five_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/5-epochs_-1")
path_to_c_five_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/5-epochs-contrl_-1")
path_to_seven_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/7-epochs_-1")
path_to_c_seven_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/7-epochs-contrl_-1")
path_to_nine_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/9-epochs_-1")
path_to_c_nine_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/9-epochs-contrl_-1")
path_to_eleven_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/11-epochs_-1")
path_to_c_eleven_not1_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/11-epochs-contrl_-1")

In [ ]:
path_to_1_graphs = Path("/Users/alexmedvedev/Desktop/graphs/1_graphs")
path_to_not_1_graphs = Path("/Users/alexmedvedev/Desktop/graphs/not_1_graphs")
path_to_80_graphs = Path("/Users/alexmedvedev/Desktop/graphs/80_graphs")

In [ ]:
auditory_temporal = ['TP9', 'TP7', 'T7', 'FT7', 'TP10', 'TP8', 'T8', 'FT8']
math_and_auditory_frontal = ['F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8'] # потом посмотри подругому, но пока так
math_and_auditory_parietal = ['P7', 'P5', 'P1', 'Pz', 'P2', 'P4', 'P8'] # потом посмотри подругому, но пока так # НАДО БУДЕТ ПОТОМ КАЖДЫЙ РАЗ УБИРАТЬ ОТ СЮДА 'BADS': 'P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8'
math_and_auditory_frontoparietal = ['F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8', 'P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8']

вроде поломанные айдишки у всех файлов поломаны данным образом
'New Segment/': 1,
 'Noise': 2,
 'Stimulus/S  5': 3,
 'Stimulus/S  7': 4,
 'Stimulus/S  9': 5,
 'Stimulus/S 11': 6,
 'Stimulus/S 25': 7,
 'Stimulus/S 27': 8,
 'Stimulus/S 29': 9,
 'Stimulus/S 35': 10,
 'Stimulus/S 51': 11,
 'Stimulus/S 63': 12,
 'Stimulus/S 71': 13,
 'Stimulus/S 91': 14,
 'Stimulus/S111': 15,
 'Stimulus/S211': 16

# Старый код

In [ ]:
file_list = list(path_to_exp_epoches.glob("*_epo.fif"))
file_list

In [ ]:
all_epochs = []

In [ ]:
all_bad_channels = set()

In [ ]:
# Loop through each file and process it
for file in file_list:

    # Load the epochs
    epochs = mne.read_epochs(file, preload=True)
    epochs = epochs["11-12"] # сдесь писать интересующую группу стимулов '5-6' и т.д. или 'contr_5-6' и т.д. + УБИРАТЬ ЭТУ СТРОЧКУ ПРИ АНАЛИЗЕ ЭПОХ ДО СТИМУЛА
    
    # Select only EEG channels (if necessary)
    picks = mne.pick_types(epochs.info, meg=False, eeg=True, eog=False, stim=False)

    # Add the bad channels from this file to the set
    all_bad_channels.update(epochs.info['bads'])
    
    # Append the epochs to the list
    all_epochs.append(epochs)

In [ ]:
# Set the union of all bad channels to each epochs object
for epochs in all_epochs:
    epochs.info['bads'] = list(all_bad_channels)

In [ ]:
# Concatenate all the epochs
all_epochs = mne.concatenate_epochs(all_epochs)

# Новый GPT анализ (хорошо работает, мне нравится больше всего) !!! делать этот анализ сразу после Prep

In [ ]:
all_dis_2 = [file for file in path_to_c_nine_not1_epoches.iterdir() if file.name.endswith('fif')]
all_dis_2

In [ ]:
all_dis_epochs_2 = {file.name: None for file in all_dis_2}

# Read epochs and store them in the dictionary
for file in all_dis_2:
    dis_epochs_2 = mne.read_epochs(file, preload=True)
    all_dis_epochs_2[file.name] = dis_epochs_2

In [ ]:
# Parameters for time-frequency analysis
frequencies = np.arange(1, 80, 1)  # Frequency range # ЕСЛИ ХОЧЕШЬ ЧТОБЫ ПРОГРАММА ГРУЗИЛА БЫСТРЕЕ УБЕРИ STEP!!!
n_cycles = frequencies / 2  # Different number of cycles per frequency

# Calculate power and inter-trial coherence (ITC)
all_power = {}
all_itc = {}

In [ ]:
for event_id in all_dis_epochs_2[next(iter(all_dis_epochs_2))].event_id.keys():
    power_list = []
    itc_list = []
    
    for epochs in all_dis_epochs_2.values():
        power, itc = mne.time_frequency.tfr_morlet(epochs[event_id], freqs=frequencies, n_cycles=n_cycles, return_itc=True, picks = math_and_auditory_frontal)
        power_list.append(power)
        itc_list.append(itc)
    
    # Average the power and ITC across subjects
    grand_average_power = mne.grand_average(power_list)
    grand_average_itc = mne.grand_average(itc_list)
    
    all_power[event_id] = grand_average_power
    all_itc[event_id] = grand_average_itc

In [ ]:
for event_id, grand_average_power in all_power.items():
    figs = grand_average_power.plot([0], baseline = (None, 0), mode='logratio', title=f'Grand Average Power frontal lobes for Event ID {event_id}', vlim = (0.5, -0.5))

    for fig in figs:
        # Access the axis from the TFR plot
        ax = fig.axes[0]

        # Add vertical lines at specific time points
        ax.axvline(x=0, color='r', linestyle='--', linewidth=2, label='Stimulus Onset')
        print (event_id)
        x_position = -0.74 # -0.53 -0.62 -0.74 -0.88 МЕНЯТЬ В ЗАВИСИМОСТИ ОТ ЭКСПЕРИМЕНТАЛЬНОЙ ГРУППЫ 
        ax.axvline(x=x_position, color='r', linestyle='--', linewidth=2)

    # Save the figure
    filename = f'Grand Average Power frontal lobes for Event ID {event_id}'
    plt.savefig(path_to_80_graphs/filename, dpi=300)  # Save with high resolution + If not one path= path_to_not_1_graphs

# ВСЁ, ЧТО ПОД ДВУМЯ СЛЕШАМИ - НОРМ КОД

    # Close the figure to avoid display during loops
    #plt.close(fig)

    # Plotting topomap for specific times and frequencies
    ##times = [0.1, 0.5]  # Specific times in seconds to plot [-2.25, -1.75]='-5-6'   [-2.5, -2.0]='-7-8'   [-2.75, -2.25]='-9-10'   [-2.9, -2.4]='-11-12' normal: [0.1, 0.5]
    ##freqs = [0.5, 4, 8, 12, 35]  # Specific frequencies in Hz to plot
    ##freq_plus = [3.5, 4, 4, 13, 45]


    ##for time in times:
        ##for i in range(len(freqs)):
            ##fig_topo = grand_average_power.plot_topomap(
                ##tmin=time, tmax=time + 0.4, fmin=freqs[i], fmax=freqs[i] + freq_plus[i], baseline=(None, 0),
                ##mode='logratio', ch_type='eeg', cmap='RdBu_r', contours=4)
            
            ##plt.suptitle(f'Topomap at {freqs[i]} Hz, {time} s for Event ID {event_id}')

            # Save the figure
            ##filename = f'topomap_{event_id}_{freqs[i]}Hz_{time}s.png'
            ##plt.savefig(path_to_1_graphs/filename, dpi=300)  # Save with high resolution + If not one path= path_to_not_1_graphs

            # Close the figure to avoid display during loops
            #plt.close(fig_topo)

# W-test

In [ ]:
all_dis_2 = [file for file in path_to_80_all.iterdir() if file.name.endswith('fif')]
all_dis_2

In [ ]:
# Read epochs and store them in the dictionary
epochs_list = []
for file in all_dis_2:
    dis_epochs_2 = mne.read_epochs(file, preload=True)
    selected_epochs = dis_epochs_2['7-8']
    epochs_list.append(selected_epochs)

# Assume you have a list of epochs
common_bads = list(set(epochs_list[0].info['bads']).union(*[ep.info['bads'] for ep in epochs_list]))

for ep in epochs_list:
    ep.info['bads'] = common_bads

epochs_cleaned = [ep.copy().drop_channels(ep.info['bads']) for ep in epochs_list]
combined_epochs = mne.concatenate_epochs(epochs_cleaned)

all_dis_epochs_1 = combined_epochs.average()

In [ ]:
# Read epochs and store them in the dictionary
epochs_list = []
for file in all_dis_2:
    dis_epochs_2 = mne.read_epochs(file, preload=True)
    selected_epochs = dis_epochs_2['contr_7-8']
    epochs_list.append(selected_epochs)

# Assume you have a list of epochs
common_bads = list(set(epochs_list[0].info['bads']).union(*[ep.info['bads'] for ep in epochs_list]))

for ep in epochs_list:
    ep.info['bads'] = common_bads

epochs_cleaned = [ep.copy().drop_channels(ep.info['bads']) for ep in epochs_list]
combined_epochs = mne.concatenate_epochs(epochs_cleaned)

all_dis_epochs_2 = combined_epochs.average()

In [ ]:
# Define frequencies of interest (log-spaced)

# НУЖНА ОДНА ИЗ ДВУХ СЛУДУЮЩИХ СТРОЧЕК:
#freqs = np.arange(1, 80, 1)
freqs = np.logspace(*np.log10([1, 80]), num=40)

n_cycles = freqs / 2.  # different number of cycles per frequency

In [ ]:
power_1 = mne.time_frequency.tfr_morlet(all_dis_epochs_1, freqs=freqs, n_cycles=n_cycles, return_itc=False, picks = math_and_auditory_parietal,  average=True)

In [ ]:
power_2 = mne.time_frequency.tfr_morlet(all_dis_epochs_2, freqs=freqs, n_cycles=n_cycles, return_itc=False, picks = math_and_auditory_parietal,  average=True)

In [ ]:
power_1

In [ ]:
power_2

In [ ]:
# Define the time and frequency window for analysis
time_window = (1.0, 1.15)  # in seconds
freq_window = (60, 70)  # in Hz

In [ ]:
# Extract the data in the selected time and frequency window
time_mask_1 = (power_1.times >= time_window[0]) & (power_1.times <= time_window[1])
freq_mask_1 = (power_1.freqs >= freq_window[0]) & (power_1.freqs <= freq_window[1])

time_mask_2 = (power_2.times >= time_window[0]) & (power_2.times <= time_window[1])
freq_mask_2 = (power_2.freqs >= freq_window[0]) & (power_2.freqs <= freq_window[1])

In [ ]:
# Extract the power data in the selected time-frequency window for each condition and channel
data_condition_1 = power_1.data[:, freq_mask_1, :][:, :, time_mask_1]
data_condition_2 = power_2.data[:, freq_mask_2, :][:, :, time_mask_2]

In [ ]:
# Reshape the data to combine frequency and channel dimensions
data_condition_1 = data_condition_1.reshape(data_condition_1.shape[0], -1)
data_condition_2 = data_condition_2.reshape(data_condition_2.shape[0], -1)

In [ ]:
a = data_condition_1.mean ()
a

In [ ]:
a = data_condition_2.mean ()
a

In [ ]:
std_deviation = np.std(data_condition_1)
std_deviation

In [ ]:
std_deviation = np.std(data_condition_2)
std_deviation

In [ ]:

# Perform the Wilcoxon signed-rank test across channels
res = wilcoxon(data_condition_1.mean(axis=0), data_condition_2.mean(axis=0))

In [ ]:
res

# W-test Fisher Z

In [ ]:
all_dis_2 = [file for file in path_to_80_all.iterdir() if file.name.endswith('fif')]
all_dis_2

In [ ]:
# Read epochs and store them in the dictionary
epochs_list_1 = []
for file in all_dis_2:
    dis_epochs_2 = mne.read_epochs(file, preload=True)
    selected_epochs = dis_epochs_2['5-6']
    epochs_list_1.append(selected_epochs)

common_bads = list(set(epochs_list_1[0].info['bads']).union(*[ep.info['bads'] for ep in epochs_list_1]))

for ep in epochs_list_1:
    ep.info['bads'] = common_bads

epochs_cleaned = [ep.copy().drop_channels(ep.info['bads']) for ep in epochs_list_1]

epochs_list_1 = mne.concatenate_epochs(epochs_cleaned)

In [ ]:
# Read epochs and store them in the dictionary
epochs_list_2 = []
for file in all_dis_2:
    dis_epochs_2 = mne.read_epochs(file, preload=True)
    selected_epochs = dis_epochs_2['11-12']
    epochs_list_2.append(selected_epochs)

common_bads = list(set(epochs_list_2[0].info['bads']).union(*[ep.info['bads'] for ep in epochs_list_2]))

for ep in epochs_list_2:
    ep.info['bads'] = common_bads

epochs_cleaned = [ep.copy().drop_channels(ep.info['bads']) for ep in epochs_list_2]

epochs_list_2 = mne.concatenate_epochs(epochs_cleaned)

In [ ]:
# Step 2: Crop the epochs to the desired time window
# Specify the time window (in seconds)
tmin = 0.0  # Start time of the window
tmax = 1.5  # End time of the window

# Crop the epochs to the specified time window
epochs_list_1 = epochs_list_1.crop(tmin=tmin, tmax=tmax)
epochs_list_2 = epochs_list_2.crop(tmin=tmin, tmax=tmax)

In [ ]:
# Step 2: Compute connectivity directly in the time-frequency domain
frequencies = np.arange(12, 36, 1)  # Alpha band: 8-12 Hz
sfreq = epochs_list_2.info['sfreq']  # Sampling frequency
n_cycles = frequencies / 2.

con_condition1 = spectral_connectivity_time(
    epochs_list_1, method='coh', mode='multitaper', sfreq=sfreq, fmin=12.0, fmax=35.0,
    faverage=False, n_jobs=1, freqs=frequencies, n_cycles=n_cycles)

In [ ]:
con_condition2 = spectral_connectivity_time(
    epochs_list_2, method='coh', mode='multitaper', sfreq=sfreq, fmin=12.0, fmax=35.0,
    faverage=False, n_jobs=1, freqs=frequencies, n_cycles=n_cycles)

In [ ]:
# Extract connectivity data
con_condition1_data = con_condition1.get_data()
con_condition1_data
con_condition2_data = con_condition2.get_data()
con_condition2_data

In [ ]:
# Step 4: Apply the Fisher Z-transform to the connectivity values
fisher_z_condition1 = np.arctanh(con_condition1_data)
fisher_z_condition2 = np.arctanh(con_condition2_data)

In [ ]:
fisher_z_condition1.shape

In [ ]:
fisher_z_condition2.shape

In [ ]:
# Initialize an empty list to hold the Wilcoxon test results
wilcoxon_results = []

#if len(fisher_z_condition1.shape) == 4:
for i in range(fisher_z_condition1.shape[2]):  # Loop through frequency bins
        for j in range(fisher_z_condition1.shape[3]):  # Loop through time points
            # Perform Wilcoxon test for each frequency-time pair
            stat, p_value = wilcoxon(fisher_z_condition1[:, :, i, j].flatten(),
                                     fisher_z_condition2[:, :, i, j].flatten())
            wilcoxon_results.append((i, j, stat, p_value))
#else:
    #print("Unexpected shape of connectivity data:", fisher_z_condition1.shape)

In [ ]:
# Convert results to a more readable format
wilcoxon_results = np.array(wilcoxon_results, dtype=[('freq_index', int), ('time_index', int), ('stat', float), ('p_value', float)])
wilcoxon_results

# Further analysis: Adjust p-values for multiple comparisons if needed (e.g., using Bonferroni correction)

# И ещё один (мне кажется, что предыдущий лучше)

In [ ]:
# Define frequencies of interest (log-spaced)
freqs = np.logspace(*np.log10([1, 50]), num=40)
n_cycles = freqs / 2.  # different number of cycle per frequency

# Time-frequency decomposition using Morlet wavelets
power = mne.time_frequency.tfr_morlet(all_epochs, freqs=freqs, n_cycles=n_cycles, return_itc=False, decim=3, average=True)

# Plot the results
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

power.plot(baseline=(-0.5, 0), mode='logratio', title = 'auto', axes=ax, colorbar=True, show=False)

# Save the figure
plt.savefig('time_frequency_plot.png')

# Show the plot
plt.show()

# Продолжение старого скрипта

In [ ]:
# Compute the Power Spectral Density (PSD) and plot the average
all_epochs.compute_psd(fmin=1.0, fmax=50.0).plot(
    average=True, amplitude=False, picks="data", exclude="bads"
)

We start by exploring the frequency content of our epochs.

Let’s first check out all channel types by averaging across epochs.

Now, let’s take a look at the spatial distributions of the PSD, averaged across epochs and frequency bands.

In [ ]:
# test graph
all_epochs.compute_psd().plot_topomap(ch_type="eeg", normalize=False, contours=0)

Alternatively, you can also create PSDs from Epochs methods directly. (even though it seems pretty useless)

In [ ]:
_, ax = plt.subplots()
spectrum = epochs.compute_psd(fmin=0.0, fmax=50.0, tmax=3.0, n_jobs=None)
# average across epochs first
mean_spectrum = spectrum.average()
psds, freqs = mean_spectrum.get_data(return_freqs=True)
# then convert to dB and take mean & standard deviation across channels
psds = 10 * np.log10(psds)
psds_mean = psds.mean(axis=0)
psds_std = psds.std(axis=0)

ax.plot(freqs, psds_mean, color="k")
ax.fill_between(
    freqs,
    psds_mean - psds_std,
    psds_mean + psds_std,
    color="k",
    alpha=0.5,
    edgecolor="none",
)
ax.set(
    title="Multitaper PSD (5-6)",
    xlabel="Frequency (Hz)",
    ylabel="Power Spectral Density (dB)",
)
plt.show()

Notably, mne.Epochs.compute_psd() supports the keyword argument average, which specifies how to estimate the PSD based on the individual windowed segments. The default is average='mean', which simply calculates the arithmetic mean across segments. Specifying average='median', in contrast, returns the PSD based on the median of the segments (corrected for bias relative to the mean), which is a more robust measure.

We now compute time-frequency representations (TFRs) from our Epochs. We’ll look at power and inter-trial coherence (ITC).

To this we’ll use the function mne.time_frequency.tfr_morlet() but you can also use mne.time_frequency.tfr_multitaper() or mne.time_frequency.tfr_stockwell().

define frequencies of interest (log-spaced)

In [ ]:
freqs = np.logspace(*np.log10([1, 44]), num=8)
n_cycles = freqs / 2.0  # different number of cycle per frequency
power, itc = epochs.compute_tfr(
    method="morlet",
    freqs=freqs,
    n_cycles=n_cycles,
    average=True,
    return_itc=True,
    decim=3,
)

In [ ]:
#layout_from_raw = [mne.channels.make_eeg_layout(raw.info)]
# same result as mne.channels.find_layout(raw.info, ch_type='eeg')

Inspect power

In [ ]:
#power.plot_topo(baseline=(-0.5, 0), mode="logratio", title="Average power")
#power.plot(picks='eeg', baseline=(-0.5, 0), mode="logratio", title='auto')
fig, axes = plt.subplots(1, 5, figsize=(25, 15), layout="constrained")
topomap_kw = dict(
    ch_type="eeg", tmin=0.5, tmax=1.5, baseline=(-0.5, 0), mode="logratio", show=False
)
plot_dict = dict(Delta=dict(fmin=0, fmax=4), Theta=dict(fmin=4, fmax=8), Alpha=dict(fmin=8, fmax=12), Beta=dict(fmin=12, fmax=30), Gamma=dict(fmin=30, fmax=45))
i = 0
for ax, (title, fmin_fmax) in zip(axes, plot_dict.items()):
    power.plot_topomap(**fmin_fmax, axes=ax, **topomap_kw, vlim = (-3.5e-01, 3.5e-01)) # , vlim = (-3.0e-01, 3.0e-01)
    ax.set_title(title)

You can also create a joint plot showing both the aggregated TFR across channels and topomaps at specific times and frequencies to obtain a quick overview regarding oscillatory effects across time and space.

In [ ]:
power.plot_joint(
    baseline=(-0.5, 0), mode="mean", tmin=-0.5, tmax=2, timefreqs=[(0.4, 5), (0.1, 5)]
)

Inspect ITC

In [ ]:
itc.plot_topo(title="Inter-Trial coherence", cmap="Reds")